In [1]:
import sys
import os

# Ruta ubicacion scripts mhw
ruta_mhw = os.path.abspath(os.path.join('..', 'mhw'))

# Agregamos la ruta al sistema
if ruta_mhw not in sys.path:
    sys.path.append(ruta_mhw)

import warnings
# Ignora warnings especificamente de numpy y la existencia de arrays con nans
#warnings.filterwarnings("ignore", message="All-NaN slice encountered")

%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import xarray as xr
import pandas as pd

import datasets_utils as dsu
import mhw_core as mhw
import time_utils as timeu

nf_waves = "pcaso_waves.nc"
nf_df_waves = "pcaso_df_waves.csv"

In [3]:
if dsu.exist_file(dsu.join_path(dsu.DIR_DATASETS, nf_waves)):
    waves = xr.open_dataset(dsu.join_path(dsu.DIR_DATASETS, nf_waves))
else:
    waves = mhw.get_marine_heat_waves(ds_clim.is_sup_p90)
    dsu.save_dataset_as_nc(waves, name_file=nf_waves)

if dsu.exist_file(dsu.join_path(dsu.DIR_DATASETS, nf_df_waves)):
    df_waves = pd.read_csv(dsu.join_path(dsu.DIR_DATASETS, nf_df_waves))
else:
    df_waves = mhw.waves_to_dataframe(waves.waves, anomalies=ds_clim.anomalias)
    dsu.save_dataframe_as_csv(df, name_file=nf_df_waves)

year_init = waves.time.min().dt.year.item() 
year_end = waves.time.max().dt.year.item()

In [4]:
df_waves["year"] = df_waves.t0.apply(timeu.extract_year_from_str_date)

In [9]:
df_waves.columns

Index(['lat', 'lon', 't0', 'tf', 'duration', 'max_anom', 'min_anom',
       'mean_anom', 'std_anom', 'acc_anom', 'growth_anom', 'decline_anom',
       'year'],
      dtype='object')

In [12]:
# 1. Agrupación y aplanado de columnas (tal como lo tenías, que está perfecto)
gv = df_waves.groupby(["year", "lat", "lon"]).agg(
    {
        "duration": ["mean", "count"],
        "max_anom": ["max", "mean", "min"],
        "min_anom": ["mean", "max", "min"],
        "acc_anom": ["mean", "max", "std", "sum"],
        "growth_anom": ["mean", "max"],
        "decline_anom": ["mean", "min", "max"]
    }
)
# 2. Renombrar columnas dataframe agrupado
gv.columns = [f"{c1}_{c2}" for c1, c2 in gv.columns]

# 3. Transformo mi dataframe agrupado en un dataset 
ds_anual_stats = gv.to_xarray()

In [15]:
ds_anual_stats = ds_anual_stats.rename({
    "duration_count": "total_events",
    "duration_mean": "duration",
    "max_anom_max": "anom_max",
    "max_anom_mean": "mean_of_max_anomalies",
    "min_anom_mean":  "mean_of_min_anomalies",
    "acc_anom_mean": "mean_of_acc_anomalies",
    "acc_anom_max": "max_of_acc_anomalies",
    "acc_anom_std": "variation_of_acc_anomalies",
    "acc_anom_sum": "total_acc_anomalies"}
)

In [17]:
# dsu.save_dataset_as_nc(ds_anual_stats, name_file="pcaso_anual_stats.nc")